# 📘 Clase 21: Gradient Boosting — modelos potentes

**Idea central:** El Gradient Boosting construye cada árbol para corregir los errores del anterior, logrando una de las mayores precisiones disponibles para datos tabulares.

En esta clase compararemos cuatro clasificadores sobre el mismo dataset y veremos por qué el GBM suele ganar en problemas con datos tabulares.

## 📦 Celda 1 — Importar librerías y preparar datos

In [ ]:
# Qué hace: importar todos los modelos y cargar el dataset de estudiantes.
# Para qué sirve: tener el entorno listo para la comparación justa de cuatro clasificadores.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.read_csv("../../datasets/estudiantes.csv")

# Etiqueta: 1 = Aprobado, 0 = En Riesgo
df["estado"] = ((df["evaluacion_python"] >= 60) & (df["asistencia_pct"] >= 70)).astype(int)

print("Dataset:", df.shape)
print(df["estado"].value_counts().rename({0: "En Riesgo", 1: "Aprobado"}))

## ✂️ Celda 2 — Separar datos para comparación justa

In [ ]:
# Qué hace: definir X e y y hacer un solo split que usarán todos los modelos.
# Para qué sirve: garantizar que la comparación sea justa (mismo test para todos).
X = df[["asistencia_pct", "evaluacion_python", "evaluacion_pandas", "edad"]]
y = df["estado"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} filas  |  Test: {X_test.shape[0]} filas")

## 🚀 Celda 3 — GradientBoostingClassifier: el concepto

Cada árbol aprende a corregir los errores residuales de los árboles anteriores.

In [ ]:
# Qué hace: entrenar el GradientBoostingClassifier con parámetros estándar.
# Para qué sirve: mostrar el modelo estrella de la clase y sus parámetros clave.
gbm = GradientBoostingClassifier(
    n_estimators=100,   # cuántos árboles construir en secuencia
    learning_rate=0.1,  # cuánto corrige cada árbol (paso de aprendizaje)
    max_depth=3,        # árboles poco profundos para no sobreajustar
    random_state=42
)
gbm.fit(X_train, y_train)

print("GradientBoostingClassifier")
print(f"  Accuracy en TRAIN: {gbm.score(X_train, y_train):.3f}")
print(f"  Accuracy en TEST:  {gbm.score(X_test,  y_test):.3f}")

## ⚡ Celda 4 — Efecto del learning_rate

¿Qué pasa si el paso de aprendizaje es demasiado grande o demasiado pequeño?

In [ ]:
# Qué hace: comparar tres GBM con distintos learning_rate.
# Para qué sirve: entender el trade-off entre velocidad de aprendizaje y sobreajuste.
tasas = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
acc_train_vals, acc_test_vals = [], []

for lr in tasas:
    m = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    m.fit(X_train, y_train)
    acc_train_vals.append(m.score(X_train, y_train))
    acc_test_vals.append(m.score(X_test,  y_test))

plt.figure(figsize=(8, 4))
plt.plot(tasas, acc_train_vals, marker="o", label="Train", color="steelblue")
plt.plot(tasas, acc_test_vals,  marker="s", label="Test",  color="darkorange")
plt.xlabel("learning_rate")
plt.ylabel("Accuracy")
plt.title("Efecto del learning_rate (n_estimators=100)")
plt.legend()
plt.tight_layout()
plt.show()
print("Observación: con learning_rate muy alto, Train sube pero Test puede bajar (sobreajuste).")

## 🧮 Celda 5 — Entrenar los cuatro clasificadores

In [ ]:
# Qué hace: entrenar LogisticRegression, DecisionTree, RandomForest y GBM.
# Para qué sirve: tener todos los modelos listos para la comparación final.
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Árbol (d=3)":         DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    resultados[nombre] = {
        "Train": round(modelo.score(X_train, y_train), 3),
        "Test":  round(modelo.score(X_test,  y_test),  3)
    }

print("Modelos entrenados correctamente.")

## 📊 Celda 6 — Tabla comparativa de accuracy

In [ ]:
# Qué hace: mostrar en una tabla clara el accuracy train y test de los cuatro modelos.
# Para qué sirve: identificar de un vistazo cuál modelo generaliza mejor.
df_res = pd.DataFrame(resultados).T.reset_index().rename(columns={"index": "Modelo"})
print(df_res.to_string(index=False))
print("\n💡 El mejor modelo en TEST es el que tiene mayor accuracy sin alejarse mucho del Train.")

## 📈 Celda 7 — Gráfico comparativo visual

In [ ]:
# Qué hace: graficar barras comparativas de accuracy en test para los cuatro modelos.
# Para qué sirve: comunicar los resultados de forma clara y visual.
nombres = list(resultados.keys())
accs_test = [resultados[n]["Test"] for n in nombres]

colores = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759"]
plt.figure(figsize=(8, 4))
bars = plt.bar(nombres, accs_test, color=colores)
for bar, val in zip(bars, accs_test):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             str(val), ha="center", va="bottom", fontsize=10)
plt.ylim(0, 1.05)
plt.ylabel("Accuracy en TEST")
plt.title("Comparación de 4 clasificadores")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 🔬 Celda 8 — Reporte detallado del mejor modelo

In [ ]:
# Qué hace: imprimir el classification_report del Gradient Boosting.
# Para qué sirve: ver precisión, recall y F1 por clase, no solo el accuracy global.
y_pred_gbm = modelos["Gradient Boosting"].predict(X_test)

print("Reporte detallado — Gradient Boosting:")
print(classification_report(y_test, y_pred_gbm, target_names=["En Riesgo", "Aprobado"]))

## ⚡ Celda 9 — XGBoost: la versión turbo del Gradient Boosting

In [ ]:
# Qué hace: intentar entrenar XGBClassifier con los mismos parámetros.
# Para qué sirve: introducir XGBoost, la librería más usada en competencias Kaggle.
try:
    import xgboost as xgb
    modelo_xgb = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    modelo_xgb.fit(X_train, y_train)
    print(f"XGBoost accuracy en test: {modelo_xgb.score(X_test, y_test):.3f}")
    print("Comparación — sklearn GBM vs XGBoost:")
    print(f"  sklearn GBM: {modelos['Gradient Boosting'].score(X_test, y_test):.3f}")
    print(f"  XGBoost:     {modelo_xgb.score(X_test, y_test):.3f}")
except ImportError:
    print("XGBoost no está instalado. Ejecuta: pip install xgboost")
    print("La diferencia principal con sklearn GBM es mayor velocidad y regularización incorporada.")

## ✅ Celda 10 — Resumen: cuándo usar cada modelo

In [ ]:
# Qué hace: imprimir una guía de decisión para elegir el modelo correcto.
# Para qué sirve: consolidar el criterio de selección de modelos del estudiante.
guia = pd.DataFrame({
    "Modelo": ["Regresión Logística", "Árbol de decisión", "Random Forest", "Gradient Boosting"],
    "Fortaleza": [
        "Muy interpretable, rápido",
        "Interpretable, visualizable",
        "Robusto, poco sobreajuste",
        "Alta precisión en datos tabulares"
    ],
    "Úsalo cuando": [
        "Necesitas explicar el modelo a alguien no técnico",
        "Quieres ver exactamente qué preguntas hace el modelo",
        "Quieres precisión sin mucho tuning",
        "Quieres la máxima precisión y tienes tiempo para afinar"
    ]
})
print(guia.to_string(index=False))
print("\n🏆 GBM y XGBoost son los modelos más usados en competencias Kaggle de datos tabulares.")